# Telegram Shop Bot — Google Colab

ربات فروشگاه مکمل‌های ورزشی (aiogram 3 + SQLAlchemy + SQLite)

**قبل از شروع:**
1. از [@BotFather](https://t.me/BotFather) توکن ربات بگیرید.
2. شناسه عددی تلگرام خود را از [@userinfobot](https://t.me/userinfobot) بگیرید (برای `/admin`).
3. پوشه پروژه را به صورت **ZIP** آماده کنید یا روی Google Drive بگذارید.

**محدودیت Colab:** اگر تب را ببندید یا زمان اجرا تمام شود، ربات قطع می‌شود. برای استفاده دائمی بهتر است روی VPS یا کامپیوتر خودتان اجرا کنید.

## ۱) نصب کتابخانه‌ها و ابزار Tunnel

In [1]:
!pip install -q -r requirements.txt nest_asyncio
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirements.txt'


## ۲) آپلود پروژه

**روش A — آپلود ZIP (ساده‌تر):** سلول بعدی را اجرا کنید و فایل zip پروژه را انتخاب کنید.

**روش B — Google Drive:** در سلول بعدی `USE_DRIVE = True` بگذارید و مسیر پوشه پروژه را تنظیم کنید.

In [5]:
import os
import zipfile
from pathlib import Path

USE_DRIVE = False  # True = استفاده از Drive
DRIVE_PROJECT_PATH = "/content/drive/MyDrive/TelegramShopBot/Cursor Ai-T_1"

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    project_dir = Path(DRIVE_PROJECT_PATH)
    if not project_dir.exists():
        raise FileNotFoundError(f"پوشه پیدا نشد: {project_dir}")
else:
    from google.colab import files
    print("فایل ZIP پروژه را انتخاب کنید (شامل main.py و پوشه bot و db و services):")
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("هیچ فایلی آپلود نشد.")
    zip_name = list(uploaded.keys())[0]
    extract_to = Path("/content/shop_bot")
    extract_to.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_name, "r") as zf:
        zf.extractall(extract_to)
    # اگر zip یک پوشه والد دارد، به داخلش برو
    candidates = [extract_to, *extract_to.iterdir()]
    project_dir = None
    for c in candidates:
        if c.is_dir() and (c / "main.py").exists():
            project_dir = c
            break
    if project_dir is None:
        raise FileNotFoundError("main.py در zip پیدا نشد. کل پوشه پروژه را zip کنید.")

os.chdir(project_dir)
print("مسیر پروژه:", project_dir)
print("فایل‌ها:", [p.name for p in Path(".").iterdir()])

فایل ZIP پروژه را انتخاب کنید (شامل main.py و پوشه bot و db و services):


KeyboardInterrupt: 

## ۳) تنظیم توکن، ادمین و تنظیمات پرداخت

In [3]:
import os

# ↓↓↓ این دو مقدار را پر کنید ↓↓↓
BOT_TOKEN = "8677860040:AAHfcQ1i30TjJ9xjM_vDagaSQ7lOb6jFowg"
ADMIN_ID = 123456789  # شناسه عددی تلگرام شما

os.environ["BOT_TOKEN"] = BOT_TOKEN.strip()
os.environ["ADMIN_IDS"] = str(ADMIN_ID)
os.environ["DATABASE_URL"] = "sqlite+aiosqlite:///shop.db"
os.environ["PAYMENT_GATEWAY"] = "mock"
os.environ["PAYMENT_SERVER_HOST"] = "0.0.0.0"
os.environ["PAYMENT_SERVER_PORT"] = "8000"

if "PASTE_YOUR" in os.environ["BOT_TOKEN"]:
    raise ValueError("لطفاً BOT_TOKEN را از BotFather وارد کنید.")

## ۴) ساخت آدرس عمومی callback برای پرداخت

این سلول یک URL عمومی با Cloudflare Tunnel می‌سازد و در `PAYMENT_CALLBACK_BASE_URL` قرار می‌دهد.

In [4]:
# import re
# import time
# import subprocess
# import urllib.request

# port = os.environ.get("PAYMENT_SERVER_PORT", "8000")

# # اجرای tunnel در بک‌گراند
# _ = subprocess.Popen(
#     ["/content/cloudflared", "tunnel", "--url", f"http://127.0.0.1:{port}"],
#     stdout=subprocess.PIPE,
#     stderr=subprocess.STDOUT,
#     text=True,
# )

# public_url = None
# for _ in range(40):
#     time.sleep(1)
#     try:
#         data = urllib.request.urlopen("http://127.0.0.1:2000/metrics").read().decode("utf-8", errors="ignore")
#         match = re.search(r"https://[a-zA-Z0-9\-]+\.trycloudflare\.com", data)
#         if match:
#             public_url = match.group(0)
#             break
#     except Exception:
#         pass

# if not public_url:
#     raise RuntimeError("Public URL پیدا نشد. سلول را یک‌بار دیگر اجرا کنید.")

# os.environ["PAYMENT_CALLBACK_BASE_URL"] = public_url
# print("PAYMENT_CALLBACK_BASE_URL =", public_url)

RuntimeError: Public URL پیدا نشد. سلول را یک‌بار دیگر اجرا کنید.

In [ ]:
import subprocess

# Stop any running cloudflared processes to ensure a clean start
print("Stopping any existing cloudflared processes...")
subprocess.run(['killall', 'cloudflared'], capture_output=True, text=True)
print("Finished stopping processes. Please re-run cell `1d5cafa4` now.")

After you've re-run cell `1d5cafa4`, please run the next cell to inspect the `cloudflared` metrics.

In [ ]:
import re
import time
import urllib.request

port = os.environ.get("PAYMENT_SERVER_PORT", "8000")

print(f"Attempting to fetch metrics from http://127.0.0.1:2000/metrics (for port {port})...")

try:
    # Give cloudflared a moment to start up and expose metrics
    time.sleep(5)
    data = urllib.request.urlopen("http://127.0.0.1:2000/metrics").read().decode("utf-8", errors="ignore")
    print("--- Cloudflared Metrics Output ---")
    print(data)
    print("----------------------------------")

    match = re.search(r"https://[a-zA-Z0-9\-]+\.trycloudflare\.com", data)
    if match:
        public_url_found = match.group(0)
        print(f"Public URL found by regex: {public_url_found}")
    else:
        print("Regex did NOT find a public URL in the metrics output.")
        print("This might indicate an issue with the cloudflared process or the expected URL pattern.")
except Exception as e:
    print(f"Failed to access cloudflared metrics endpoint: {e}")
    print("This could mean cloudflared is not running or not exposing the metrics port.")

print("If no URL was found, please review the metrics output above for any unusual patterns or error messages.")

## ۵) تست دیتابیس (اختیاری)

In [ ]:
!python -m db.smoke_check

## ۶) اجرای ربات

این سلول را اجرا کنید و **تب Colab را باز نگه دارید**.

برای توقف: دکمه Stop یا Runtime -> Interrupt execution.

در تلگرام: `/start` — منوی فروشگاه | `/admin` — پنل ادمین (فقط با ADMIN_ID)

In [ ]:
import asyncio
import nest_asyncio

nest_asyncio.apply()

from main import main

print("ربات در حال اجراست... تب Colab را نبندید.")
asyncio.run(main())